# Module 0.2: Why LLM Inference Is Different

Explore why serving LLMs is fundamentally harder than traditional ML — memory-bound decode, arithmetic intensity, and the roofline model.


In [ ]:
import sys
sys.path.insert(0, '../../../labs')

import numpy as np
import matplotlib.pyplot as plt

# Import our roofline utilities
from utils.roofline import (
    RooflineModel, GPU_SPECS, plot_roofline, 
    compare_gpus, print_gpu_specs
)

---
## Part 1: Model Architecture (Llama 3.1 8B)

Let's define the concrete dimensions we'll use throughout:

In [ ]:
# Llama 3.1 8B dimensions
D_MODEL = 4096       # Hidden dimension
N_HEADS = 32         # Number of attention heads  
N_KV_HEADS = 8       # Number of KV heads (GQA)
D_HEAD = 128         # Dimension per head (4096/32)
N_LAYERS = 32        # Number of transformer layers
D_FF = 14336         # FFN intermediate dimension
PARAMS_B = 8.03      # Total parameters in billions

print("="*60)
print("LLAMA 3.1 8B ARCHITECTURE")
print("="*60)
print(f"Hidden dimension (d_model):     {D_MODEL:,}")
print(f"Attention heads:                {N_HEADS}")
print(f"KV heads (GQA):                 {N_KV_HEADS}")
print(f"Head dimension:                 {D_HEAD}")
print(f"Layers:                         {N_LAYERS}")
print(f"FFN dimension:                  {D_FF:,}")
print(f"Total parameters:               {PARAMS_B}B")
print(f"Model size (FP16):              {PARAMS_B * 2:.1f} GB")

---
## Part 2: FLOPs Calculation

Let's calculate the exact FLOPs for prefill and decode phases.

In [ ]:
def calculate_prefill_flops(seq_len: int, d_model: int = D_MODEL, 
                            n_layers: int = N_LAYERS, d_ff: int = D_FF) -> dict:
    """
    Calculate FLOPs for prefill phase.
    
    Args:
        seq_len: Number of input tokens
        d_model: Hidden dimension
        n_layers: Number of transformer layers
        d_ff: FFN intermediate dimension
    
    Returns:
        Dictionary with FLOPs breakdown
    """
    # Per layer calculations
    # QKV projection: 3 * (2 * seq_len * d_model * d_model)
    qkv_flops = 3 * 2 * seq_len * d_model * d_model
    
    # Attention scores: Q @ K^T -> [seq_len, seq_len]
    attn_scores_flops = 2 * seq_len * d_model * seq_len
    
    # Attention output: scores @ V -> [seq_len, d_model]
    attn_output_flops = 2 * seq_len * seq_len * d_model
    
    # Output projection
    output_proj_flops = 2 * seq_len * d_model * d_model
    
    # FFN (SwiGLU: gate, up, down projections)
    ffn_flops = 3 * 2 * seq_len * d_model * d_ff
    
    # Total per layer
    layer_flops = qkv_flops + attn_scores_flops + attn_output_flops + output_proj_flops + ffn_flops
    
    # Total for model
    total_flops = n_layers * layer_flops
    
    return {
        'seq_len': seq_len,
        'qkv_flops_per_layer': qkv_flops,
        'attention_flops_per_layer': attn_scores_flops + attn_output_flops,
        'output_proj_flops_per_layer': output_proj_flops,
        'ffn_flops_per_layer': ffn_flops,
        'total_flops_per_layer': layer_flops,
        'total_flops': total_flops,
        'total_tflops': total_flops / 1e12,
    }


def calculate_decode_flops(seq_len: int, d_model: int = D_MODEL,
                           n_layers: int = N_LAYERS, d_ff: int = D_FF) -> dict:
    """
    Calculate FLOPs for ONE decode step.
    
    Args:
        seq_len: Current sequence length (context)
        d_model: Hidden dimension
        n_layers: Number of transformer layers
        d_ff: FFN intermediate dimension
    
    Returns:
        Dictionary with FLOPs breakdown
    """
    # Per layer calculations (processing 1 token)
    # QKV projection for 1 token
    qkv_flops = 3 * 2 * 1 * d_model * d_model
    
    # Attention scores: Q @ K^T -> [1, seq_len]
    attn_scores_flops = 2 * 1 * d_model * seq_len
    
    # Attention output: scores @ V -> [1, d_model]
    attn_output_flops = 2 * 1 * seq_len * d_model
    
    # Output projection
    output_proj_flops = 2 * 1 * d_model * d_model
    
    # FFN for 1 token
    ffn_flops = 3 * 2 * 1 * d_model * d_ff
    
    # Total per layer
    layer_flops = qkv_flops + attn_scores_flops + attn_output_flops + output_proj_flops + ffn_flops
    
    # Total for model
    total_flops = n_layers * layer_flops
    
    return {
        'seq_len': seq_len,
        'qkv_flops_per_layer': qkv_flops,
        'attention_flops_per_layer': attn_scores_flops + attn_output_flops,
        'output_proj_flops_per_layer': output_proj_flops,
        'ffn_flops_per_layer': ffn_flops,
        'total_flops_per_layer': layer_flops,
        'total_flops': total_flops,
        'total_gflops': total_flops / 1e9,
    }

In [ ]:
# Calculate for 1000-token prompt
prefill = calculate_prefill_flops(seq_len=1000)
decode = calculate_decode_flops(seq_len=1000)

print("="*70)
print("PREFILL: 1000 tokens")
print("="*70)
print(f"QKV projection per layer:    {prefill['qkv_flops_per_layer']/1e9:>10.2f} GFLOPs")
print(f"Attention per layer:         {prefill['attention_flops_per_layer']/1e9:>10.2f} GFLOPs")
print(f"Output projection per layer: {prefill['output_proj_flops_per_layer']/1e9:>10.2f} GFLOPs")
print(f"FFN per layer:               {prefill['ffn_flops_per_layer']/1e9:>10.2f} GFLOPs")
print(f"-" * 50)
print(f"Total per layer:             {prefill['total_flops_per_layer']/1e9:>10.2f} GFLOPs")
print(f"Total (32 layers):           {prefill['total_tflops']:>10.2f} TFLOPs")

print()
print("="*70)
print("DECODE: 1 token (with 1000 tokens in context)")
print("="*70)
print(f"QKV projection per layer:    {decode['qkv_flops_per_layer']/1e6:>10.2f} MFLOPs")
print(f"Attention per layer:         {decode['attention_flops_per_layer']/1e6:>10.2f} MFLOPs")
print(f"Output projection per layer: {decode['output_proj_flops_per_layer']/1e6:>10.2f} MFLOPs")
print(f"FFN per layer:               {decode['ffn_flops_per_layer']/1e6:>10.2f} MFLOPs")
print(f"-" * 50)
print(f"Total per layer:             {decode['total_flops_per_layer']/1e6:>10.2f} MFLOPs")
print(f"Total (32 layers):           {decode['total_gflops']:>10.2f} GFLOPs")

print()
print("="*70)
print("COMPARISON")
print("="*70)
print(f"Prefill (1000 tokens):  {prefill['total_tflops']:.2f} TFLOPs")
print(f"Decode (1 token):       {decode['total_gflops']:.2f} GFLOPs = {decode['total_gflops']/1000:.4f} TFLOPs")
print(f"Ratio:                  {prefill['total_flops']/decode['total_flops']:.0f}x more FLOPs in prefill")

---
## Part 3: Arithmetic Intensity

Arithmetic intensity = FLOPs / Bytes transferred. This determines whether a workload is compute-bound or memory-bound.

In [ ]:
# Model size in bytes (FP16)
model_size_bytes = PARAMS_B * 1e9 * 2  # 2 bytes per FP16 parameter
model_size_gb = model_size_bytes / 1e9

# Prefill arithmetic intensity
prefill_ai = prefill['total_flops'] / model_size_bytes

# Decode arithmetic intensity  
decode_ai = decode['total_flops'] / model_size_bytes

# A100 ridge point
a100_ridge = 312 / 2.0  # TFLOPS / TB/s = 156 FLOPs/byte

print("="*70)
print("ARITHMETIC INTENSITY ANALYSIS")
print("="*70)
print(f"Model size (FP16):           {model_size_gb:.1f} GB")
print()
print(f"Prefill (1000 tokens):")
print(f"  FLOPs:                     {prefill['total_tflops']:.2f} TFLOPs")
print(f"  Bytes read:                {model_size_gb:.1f} GB")
print(f"  Arithmetic Intensity:      {prefill_ai:.0f} FLOPs/byte")
print(f"  vs Ridge Point (156):      {prefill_ai/a100_ridge:.1f}x higher → COMPUTE-BOUND")
print()
print(f"Decode (1 token):")
print(f"  FLOPs:                     {decode['total_gflops']:.2f} GFLOPs")
print(f"  Bytes read:                {model_size_gb:.1f} GB")
print(f"  Arithmetic Intensity:      {decode_ai:.1f} FLOPs/byte")
print(f"  vs Ridge Point (156):      {decode_ai/a100_ridge:.3f}x lower → MEMORY-BOUND")

---
## Part 4: Roofline Model Visualization

The roofline model shows the relationship between arithmetic intensity and achievable performance.

In [ ]:
# Print available GPU specs
print_gpu_specs()

In [ ]:
# Generate roofline plot for A100
fig = plot_roofline(
    gpu='A100_80GB',
    model_params_b=8.0,
    prefill_tokens=1000,
    decode_batch_sizes=[1, 8, 32],
    figsize=(12, 8)
)
plt.show()

In [ ]:
# Compare different GPUs
fig = compare_gpus(
    gpus=['A100_80GB', 'H100_SXM', 'H200'],
    figsize=(14, 7)
)
plt.show()

In [ ]:
# Custom roofline with specific workloads
model = RooflineModel(GPU_SPECS['H100_SXM'])

# Add custom workloads
model.add_workload('Decode batch=1', arithmetic_intensity=1, color='#ef4444', marker='o')
model.add_workload('Decode batch=4', arithmetic_intensity=4, color='#f97316', marker='s')
model.add_workload('Decode batch=16', arithmetic_intensity=16, color='#eab308', marker='^')
model.add_workload('Decode batch=64', arithmetic_intensity=64, color='#84cc16', marker='D')
model.add_workload('Prefill N=512', arithmetic_intensity=512, color='#22c55e', marker='o', label_offset=(1.5, 0.7))
model.add_workload('Prefill N=2048', arithmetic_intensity=2048, color='#14b8a6', marker='o', label_offset=(1.5, 0.7))

fig = model.plot(
    title='H100 Roofline: Effect of Batching on Decode',
    figsize=(12, 8)
)
plt.show()

---
## Part 5: Key Takeaways

| Phase | FLOPs | Bytes Read | Arithmetic Intensity | Bottleneck |
|-------|-------|------------|---------------------|------------|
| Prefill (N=1000) | ~16 TFLOPs | ~16 GB | ~1000 FLOPs/byte | Compute |
| Decode (batch=1) | ~16 GFLOPs | ~16 GB | ~1 FLOP/byte | Memory |

**Key insights:**
1. Prefill processes N tokens with N× more FLOPs but same memory reads → high arithmetic intensity → compute-bound
2. Decode processes 1 token but still reads entire model → low arithmetic intensity → memory-bound
3. Batching decode increases arithmetic intensity, moving workload toward compute-bound
4. Different GPUs have different ridge points, affecting which workloads are memory vs compute bound